# 02 · Step-by-step exploration: NER, then targeted tone

**Scope.** German only, one block-A vote (**A-631**, Begrenzungsinitiative,
2020-09-27), every person and organisation, tone *toward* the actor only.
Each step's output is read before the next one starts.

## 1. The slice

The archive for A-631 holds every article from the 34 outlets published inside
the event window. The query filtered on outlet, dates and language, **not on
topic**. This step keeps the German rows and checks them against the pull
receipt. Nothing here prints article text, so the notebook can be committed
with its outputs.

In [1]:
import collections
import difflib
import html
import json
import re

import numpy as np
import pandas as pd

from mediapos import config as cfg, queries as q

EVENT_ID = "A-631"
LANGUAGE = "de"

query = pd.read_csv(cfg.DATA_OUT / "queries.csv").set_index("event_id").loc[EVENT_ID]
receipts = json.loads((cfg.DATA_OUT / "pull_receipts.json").read_text())
receipt = receipts[f"{query.date_from}_{query.date_to}"]
archive = cfg.REPO_ROOT / receipt["local_path"]

print(f"event    {EVENT_ID}  window {query.date_from} → {query.date_to}")
print(f"archive  {archive.relative_to(cfg.REPO_ROOT)}")
print(f"receipt  {receipt['articles_read']:,} articles, {receipt['by_language']}")

event    A-631  window 2020-08-28 → 2020-10-11
archive  data/corpus/2020-08-28_2020-10-11.tsv.xz
receipt  87,403 articles, {'de': 58649, 'fr': 28754}


In [2]:
articles = pd.DataFrame(r for r in q.read_articles(archive) if r["language"] == LANGUAGE)
articles["pubtime"] = pd.to_datetime(articles["pubtime"], utc=True)
articles["char_count"] = articles["char_count"].astype(int)

# The rows must be exactly what the receipt says was downloaded.
assert len(articles) == receipt["by_language"][LANGUAGE], len(articles)

print(f"{len(articles):,} {LANGUAGE} rows, count matches the receipt")
print(f"{articles['id'].nunique():,} distinct article ids")
print(f"published {articles.pubtime.min():%Y-%m-%d} → {articles.pubtime.max():%Y-%m-%d}")

58,649 de rows, count matches the receipt
58,587 distinct article ids
published 2020-08-27 → 2020-10-11


### One id, several rows: revisions of the same article

Some ids appear two or three times. These rows have the same outlet and
timestamp but a different `content_id` and body: they are **edited versions**
of one online article. Counting every version would weight an article by how
often it was edited, so we keep **one row per id: the longest version**, with
ties broken by `content_id` so the choice is deterministic.

In [3]:
versions = articles.groupby("id").size()
revised = articles[articles["id"].isin(versions[versions > 1].index)]
print(f"{(versions > 1).sum()} ids have several versions ({len(revised)} rows)")
print(revised["medium_code"].value_counts().to_string())

articles = (articles.sort_values(["id", "char_count", "content_id"], ascending=[True, False, True])
                    .drop_duplicates("id").reset_index(drop=True))
assert articles["id"].is_unique
print(f"\nslice: {len(articles):,} articles, one per id")

58 ids have several versions (120 rows)
medium_code
BLIO    106
ZWAO      6
NNTA      4
NNBU      2
SRF       2



slice: 58,587 articles, one per id


### Who published them, and in what form

In [4]:
(articles.groupby(["medium_code", "medium_name"]).size()
         .rename("articles").sort_values(ascending=False).reset_index())

,medium_code,medium_name,articles
0,AZM,Aargauer Zeitung / MLZ,4567
1,SGTO,tagblatt.ch,4392
2,BLIO,blick.ch,4239
3,NNBE,bernerzeitung.ch,4007
4,SRF,srf.ch,3601
5,NNBS,bazonline.ch,3175
6,NNTA,tagesanzeiger.ch,3149
7,ZWAO,20 minuten online,3145
8,NNBU,derbund.ch,3053
9,LUZ,Luzerner Zeitung,2881


In [5]:
articles["doctype_description"].value_counts().rename("articles").to_frame()

,articles
doctype_description,
Online medium,31540
Regional daily newspaper,14450
National daily newspaper,10767
Regional weekly newspaper,1830


In [6]:
articles["char_count"].describe(percentiles=[.1, .25, .5, .75, .9]).round(0).to_frame()

,char_count
count,58587.0
mean,4602.0
std,10304.0
min,86.0
10%,896.0
25%,1741.0
50%,3288.0
75%,5120.0
90%,7570.0
max,551056.0


## 2. The articles about the vote

The window holds everything the outlets published, so we keep the articles that
**name the initiative**: its common name (*Begrenzungsinitiative*), its
opponents' name (*Kündigungsinitiative*) or its official title (*massvolle
Zuwanderung*). We search the headline, subhead and body.

*Personenfreizügigkeit* is left out on purpose. It also appears in articles
about the EU framework agreement, so it would pull in more than this vote.

Naming the initiative once does not make an article *about* it. A portrait of
the cantonal CVP can mention it in passing. So we also count the mentions and
record whether the name is in the headline, which shows how central the vote
is to each article.

In [7]:
VOTE = re.compile(
    r"Begrenzungs-?initiative|Kündigungs-?initiative|massvollen? Zuwanderung", re.IGNORECASE
)

searchable = articles["head"] + "\n" + articles["subhead"] + "\n" + articles["content"]
# pandas' str.count drops a compiled pattern's IGNORECASE flag, so count with findall.
n_mentions = searchable.map(lambda s: len(VOTE.findall(s)))
vote = articles[n_mentions > 0].assign(n_vote_mentions=n_mentions[n_mentions > 0]).reset_index(drop=True)
vote["vote_in_head"] = (vote["head"] + " " + vote["subhead"]).str.contains(VOTE)

print(f"{len(vote):,} of {len(articles):,} articles name the initiative "
      f"({100 * len(vote) / len(articles):.1f}%)")
print(f"{vote['vote_in_head'].sum()} name it in the headline or subhead")
vote["n_vote_mentions"].clip(upper=5).value_counts().sort_index().rename(
    lambda n: "5+" if n == 5 else n).rename("articles").to_frame()

1,086 of 58,587 articles name the initiative (1.9%)
82 name it in the headline or subhead


,articles
n_vote_mentions,
1,544
2,182
3,127
4,73
5+,160


In [8]:
(vote.groupby("medium_code")
     .agg(articles=("id", "size"), in_head=("vote_in_head", "sum"),
          median_mentions=("n_vote_mentions", "median"))
     .join(articles.groupby("medium_code").size().rename("all_articles"))
     .assign(share_pct=lambda t: (100 * t.articles / t.all_articles).round(1))
     .sort_values("articles", ascending=False))

,articles,in_head,median_mentions,all_articles,share_pct
medium_code,,,,,
SGTO,218,9,3.0,4392,5.0
AZM,79,0,1.0,4567,1.7
SRF,64,12,3.0,3601,1.8
NNBE,60,12,1.5,4007,1.5
NNBS,57,12,2.0,3175,1.8
SGT,53,0,1.0,2794,1.9
NNBU,53,12,2.0,3053,1.7
NNTA,50,11,2.0,3149,1.6
LUZ,48,1,1.0,2881,1.7


## 3. From markup to text

`content` is not plain text. It is XML-like markup, and each tag needs a decision:

| Tag | What it is | Decision |
|---|---|---|
| `<ld>` | lead paragraph | **body** |
| `<p>` | paragraph | **body** |
| `<zt>` | crosshead (subheading) | **body**, on its own line |
| `<a>`, `<br>`, tables | links, line breaks, cells | **body**, the markup removed and the text kept |
| `<lg>` | photo caption | own column `captions` |
| `<ka>` | box, often a pull quote that repeats the body | own column `boxes` |
| `<nt>` | note, e.g. a line about the author | own column `notes` |
| `<au>` | author byline | **dropped** |
| `<ur>` | photo credit | **dropped** |

Bylines and photo credits are dropped because NER would read the journalist's
or photographer's name as an actor in the story. Captions, boxes and notes are
kept apart rather than thrown away. Captions are visibility too (a photo of an
actor), and pull quotes would count the same mention twice if merged into the
body. Whether to use them is a later decision.

The model input `text` is headline, subhead and body, separated by blank
lines.

In [9]:
SIDE_COLUMNS = {"lg": "captions", "ka": "boxes", "nt": "notes"}
DROPPED = ("au", "ur")
LINE_BREAK = re.compile(r"</?(?:p|zt|ld|tx|br|tr|table)\b[^>]*>|</t[dh]>")
ANY_TAG = re.compile(r"<[^>]+>")


def element(tag: str) -> re.Pattern[str]:
    return re.compile(rf"<{tag}\b[^>]*>(.*?)</{tag}>", re.DOTALL)


def to_text(markup: str) -> str:
    """Markup fragment to plain text: one line per block, whitespace collapsed."""
    # Inline tags become a space: "einen<a ...>Bausatz" must not fuse into one word.
    text = html.unescape(ANY_TAG.sub(" ", LINE_BREAK.sub("\n", markup)))
    lines = (re.sub(r"\s+", " ", line).strip() for line in text.split("\n"))
    return "\n".join(line for line in lines if line)


def split_content(markup: str) -> dict[str, str]:
    # Drop first: photo credits also sit inside captions.
    for tag in DROPPED:
        markup = element(tag).sub(" ", markup)
    parts = {}
    for tag, column in SIDE_COLUMNS.items():
        parts[column] = "\n".join(to_text(m) for m in element(tag).findall(markup))
        markup = element(tag).sub("\n", markup)
    parts["body"] = to_text(markup)
    return parts


vote = vote.join(pd.DataFrame(vote["content"].map(split_content).tolist()))
vote["text"] = [
    "\n\n".join(p for p in (h.strip(), s.strip(), b) if p)
    for h, s, b in zip(vote["head"], vote["subhead"], vote["body"])
]

### Checks

The cleaning is only trusted if nothing of the markup survives and nothing of
the body is lost on the way.

In [10]:
leftover_tag = vote["text"].str.contains(r"</?[a-z]+\b[^>]*>")
leftover_entity = vote["text"].str.contains(r"&[a-z]+;|&#\d+;")
assert not leftover_tag.any(), vote.loc[leftover_tag, "id"].tolist()
assert not leftover_entity.any(), vote.loc[leftover_entity, "id"].tolist()
assert (vote["body"].str.len() > 0).all()

# Every word of the raw markup must land somewhere: body, a side column, or a
# dropped tag. Compare word counts, not strings, since whitespace changed.
words = lambda s: len(re.findall(r"\w+", s))
dropped = vote["content"].map(lambda m: sum(words(x) for t in DROPPED for x in element(t).findall(m)))
kept = vote[["body", *SIDE_COLUMNS.values()]].map(words).sum(axis=1)
raw = vote["content"].map(lambda m: words(html.unescape(ANY_TAG.sub(" ", m))))
assert (kept + dropped == raw).all(), (kept + dropped - raw).abs().max()

print("no tag or HTML entity left; every word accounted for")
print(f"articles with captions {(vote.captions != '').sum()}, boxes {(vote.boxes != '').sum()}, "
      f"notes {(vote.notes != '').sum()}")
print(f"words: body {vote.body.map(words).sum():,}, captions {vote.captions.map(words).sum():,}, "
      f"boxes {vote.boxes.map(words).sum():,}, notes {vote.notes.map(words).sum():,}, "
      f"dropped {dropped.sum():,}")

no tag or HTML entity left; every word accounted for
articles with captions 721, boxes 340, notes 25


words: body 2,829,365, captions 260,528, boxes 55,690, notes 543, dropped 2,170


In [11]:
vote["text_chars"] = vote["text"].str.len()
vote["text_chars"].describe(percentiles=[.1, .25, .5, .75, .9, .99]).round(0).to_frame()

,text_chars
count,1086.0
mean,17974.0
std,32595.0
min,381.0
10%,1638.0
25%,3119.0
50%,4805.0
75%,9148.0
90%,73211.0
99%,136561.0


### Reading one article

Run the cell below to read an article in full, before and after cleaning. **Clear
its output before committing.** Article text is licensed corpus content and
must not enter git.

In [12]:
def show(i: int) -> None:
    """Print article i of `vote` in full: header line, text, then the side columns."""
    row = vote.iloc[i]
    print(f"{row['id']}  {row['medium_name']}  {row['pubtime']:%Y-%m-%d}  "
          f"mentions={row['n_vote_mentions']}  in_head={row['vote_in_head']}  "
          f"relevance={row.get('relevance', '–')}\n")
    print(row["text"])
    for column in SIDE_COLUMNS.values():
        if row[column]:
            print(f"\n--- {column} ---\n{row[column]}")

## 4. Is the vote the main subject?

Naming the initiative is not the same as being about it. An article on the
hunting-law vote can name the initiative three times ("the same Sunday we also
vote on…"). So a model reads each article and gives **one label**:

- **related**: the main subject is the Begrenzungsinitiative or its direct
  stakes (the campaign, its actors and arguments, the result, free movement with
  the EU and the bilateral agreements as they bear on this vote).
- **not_related**: anything else, including the other four votes of
  27 September and **roundups covering several votes side by side**. In a
  roundup we cannot tell which mentions belong to the initiative.

**Model:** Claude Haiku 5.5, thinking off, low effort, with the output
constrained to the two labels. **Input:** headline, subhead and the first
4,000 characters of the body.

**Cache:** each label is saved to `data/interim/a631_relevance.jsonl` under the
article id, the model and a hash of the prompt. A rerun only calls the model for
articles with no saved label. Changing the prompt, the model or the truncation
changes the hash, so everything is labelled again. No article text is saved.

**This filter is a model and has an error rate.** It sits upstream of every
later step, so its labels get hand-checked (section 5) before they are trusted.

In [13]:
import asyncio
import hashlib

import anthropic
from dotenv import load_dotenv

load_dotenv(cfg.REPO_ROOT / ".env")

MODEL = "claude-haiku-5-5"
MAX_BODY_CHARS = 4_000
CONCURRENCY = 10
RELEVANCE_CACHE = cfg.REPO_ROOT / "data" / "interim" / "a631_relevance.jsonl"

SYSTEM = """You label Swiss German-language news articles from September and October 2020.

On 27 September 2020 Swiss voters decided five federal proposals: the Begrenzungsinitiative \
(the SVP initiative to end free movement of persons with the EU, also called \
Kündigungsinitiative), the hunting law (Jagdgesetz), the purchase of new fighter jets, \
paternity leave, and higher child tax deductions.

Label the article "related" if its main subject is the Begrenzungsinitiative or its direct \
stakes: the campaign, its actors and arguments, the result, or free movement of persons \
with the EU and the bilateral agreements as they bear on this vote.

Label it "not_related" if its main subject is anything else, even when it mentions the \
initiative. That includes articles mainly about one of the other four proposals, roundups \
that cover several of the proposals side by side, and any other topic.

Judge the main subject of the article, not whether the initiative is mentioned."""

LABEL_FORMAT = {
    "type": "json_schema",
    "schema": {
        "type": "object",
        "properties": {"label": {"type": "string", "enum": ["related", "not_related"]}},
        "required": ["label"],
        "additionalProperties": False,
    },
}

# Anything that changes the answer changes the key, so stale labels are never reused.
PROMPT_SHA = hashlib.sha256(
    json.dumps([MODEL, SYSTEM, LABEL_FORMAT, MAX_BODY_CHARS]).encode()
).hexdigest()[:12]


def classifier_input(row: pd.Series) -> str:
    return f"Headline: {row['head']}\nSubhead: {row['subhead']}\n\n{row['body'][:MAX_BODY_CHARS]}"


def load_labels() -> dict[str, dict]:
    if not RELEVANCE_CACHE.exists():
        return {}
    rows = (json.loads(line) for line in RELEVANCE_CACHE.read_text().splitlines())
    return {r["id"]: r for r in rows if r["prompt_sha"] == PROMPT_SHA}


async def label_one(client: anthropic.AsyncAnthropic, gate: asyncio.Semaphore,
                    row: pd.Series, sink) -> str | None:
    async with gate:
        try:
            response = await client.messages.create(
                model=MODEL,
                max_tokens=64,
                thinking={"type": "disabled"},
                output_config={"effort": "low", "format": LABEL_FORMAT},
                system=SYSTEM,
                messages=[{"role": "user", "content": classifier_input(row)}],
            )
        except (anthropic.APIStatusError, anthropic.APIConnectionError) as e:
            return f"{row['id']}: {type(e).__name__}"
    if response.stop_reason != "end_turn":
        return f"{row['id']}: stop_reason={response.stop_reason}"
    label = json.loads(next(b.text for b in response.content if b.type == "text"))["label"]
    sink.write(json.dumps({
        "id": row["id"], "label": label, "model": MODEL, "prompt_sha": PROMPT_SHA,
        "input_tokens": response.usage.input_tokens, "output_tokens": response.usage.output_tokens,
    }) + "\n")
    sink.flush()  # each label is on disk as soon as it exists, so an interrupted run loses nothing
    return None


async def label_missing(frame: pd.DataFrame) -> list[str]:
    todo = frame[~frame["id"].isin(load_labels().keys())]
    if todo.empty:
        return []
    RELEVANCE_CACHE.parent.mkdir(parents=True, exist_ok=True)
    gate = asyncio.Semaphore(CONCURRENCY)
    async with anthropic.AsyncAnthropic() as client:
        with RELEVANCE_CACHE.open("a") as sink:
            async with asyncio.TaskGroup() as tg:
                tasks = [tg.create_task(label_one(client, gate, row, sink)) for _, row in todo.iterrows()]
    return [t.result() for t in tasks if t.result()]

In [14]:
already = len(load_labels())
failures = await label_missing(vote)
labels = load_labels()

print(f"prompt {PROMPT_SHA}: {already} labels cached, {len(labels) - already} new, {len(failures)} failed")
for failure in failures:
    print("  ", failure)

used_in = sum(r["input_tokens"] for r in labels.values())
used_out = sum(r["output_tokens"] for r in labels.values())
print(f"tokens for all {len(labels)} labels: {used_in:,} in, {used_out:,} out "
      f"≈ ${used_in / 1e6 * 0.10 + used_out / 1e6 * 0.50:.2f} at Haiku 5.5 list price")

vote["relevance"] = vote["id"].map({i: r["label"] for i, r in labels.items()})
assert vote["relevance"].notna().all(), f"{vote['relevance'].isna().sum()} articles have no label"
vote["relevance"].value_counts().to_frame()

prompt 315452286fb2: 1086 labels cached, 0 new, 0 failed
tokens for all 1086 labels: 2,421,585 in, 11,000 out ≈ $0.25 at Haiku 5.5 list price


,count
relevance,
not_related,613
related,473


In [15]:
(vote.groupby("medium_code")
     .agg(named=("id", "size"), related=("relevance", lambda s: (s == "related").sum()))
     .assign(kept_pct=lambda t: (100 * t.related / t.named).round(0))
     .sort_values("named", ascending=False))

,named,related,kept_pct
medium_code,,,
SGTO,218,20,9.0
AZM,79,32,41.0
SRF,64,33,52.0
NNBE,60,29,48.0
NNBS,57,30,53.0
SGT,53,30,57.0
NNBU,53,28,53.0
NNTA,50,27,54.0
LUZ,48,28,58.0


In [16]:
vote.groupby("relevance")["n_vote_mentions"].describe(percentiles=[.5]).round(1)

,count,mean,std,min,50%,max
relevance,,,,,,
not_related,613.0,2.2,2.6,1.0,1.0,48.0
related,473.0,2.8,2.2,1.0,2.0,12.0


## 5. Named entities: three models

We keep the **473 articles labelled related** and run three NER models on them:

| Model | Kind | Labels |
|---|---|---|
| **spaCy** `de_core_news_lg` | classic, trained on German news | PER, ORG, LOC, MISC |
| **GLiNER** `urchade/gliner_multi-v2.1` | zero-shot, we name the labels | person, organization |
| **Claude Haiku 5.5** | prompted, returns JSON | person, organization |

The output is **one row per mention**: article, model, character position,
mention text, type. The per-article count dictionary is a tally of those rows.
The positions are kept because the tone step needs to know where each mention
stands.

Three details decide what gets counted:

- **spaCy's LOC and MISC are kept as type `other`.** That lets us see whether a
  model tags *Bern* (meaning the government) as a place. Only person and
  organization enter the counts.
- **GLiNER reads at most 384 tokens at once.** Articles are cut into pieces of
  at most 150 words, at a sentence or line end where possible so that no name
  is split, and the positions are shifted back onto the full article.
- **Haiku marks every mention inside the text** and gives it an entity id:
  *‹P1›Tiana Angelina Moser‹/P1› … ‹P1›Moser‹/P1› … die ‹O1›FDP‹/O1›-Chefin*. It
  works through the article in pieces of at most 400 words and gets the table
  of entities found so far with each piece, so *Moser* in a later piece keeps
  id P1. The ids link spellings **within** an article; linking across articles
  comes in the normalization step. A first version asked Haiku for a list of
  names and searched for them; it lost every spelling missing from the list
  (mostly surnames standing alone) and was replaced.
- **Haiku has to copy the text to tag it, and it does not always copy
  faithfully**: it corrects a case ending, adds a word, chats ("wait, I must
  copy exactly…"), and above all **skips headlines and sometimes whole
  passages**. The copy is aligned with the original and each tag carried back to
  its original position; a tag is kept only if its text stands unchanged in the
  original. Every passage the copy skipped is sent again on its own, with the
  entity table, and tagged in a second pass (a third for what that one skips).
  What remains untagged after that is counted and reported.
- **Haiku sometimes leaves a repeat untagged**: it tags *Bundesrat* in one
  sentence and not in the next. Every untagged occurrence of a form Haiku tagged
  elsewhere in the same article is filled in with that form's id (longest form
  first, whole words only). These mentions are marked `filled`, so they can
  always be counted apart or removed. A form Haiku tagged wrongly once is
  filled in wrongly too; that is the price of the fill.

Each model's output is saved in `data/interim/` under a hash of its settings, so
a rerun reads the saved mentions. The functions were tested before they went
into this cell: positions after chunking, the longest-first rule, word
boundaries, the cache.

In [17]:
import spacy
from gliner import GLiNER

related = vote[vote["relevance"] == "related"].reset_index(drop=True)
texts_by_id = dict(zip(related["id"], related["text"]))
INTERIM = cfg.REPO_ROOT / "data" / "interim"
COLUMNS = ["id", "model", "start", "end", "mention", "type", "label", "score"]


def config_sha(config: object) -> str:
    return hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:12]


def cached(name: str, config: object, run) -> pd.DataFrame:
    """Run a local model once per config; later calls read the saved mentions."""
    path = INTERIM / f"a631_ner_{name}_{config_sha(config)}.parquet"
    ids = path.with_suffix(".ids.json")
    if path.exists() and set(related["id"]) <= set(json.loads(ids.read_text())):
        saved = pd.read_parquet(path)
        return saved[saved["id"].isin(related["id"])].reset_index(drop=True)
    mentions = run(related)
    mentions.to_parquet(path, index=False)
    ids.write_text(json.dumps(sorted(related["id"])))
    return mentions


def chunks(text: str, max_words: int) -> list[tuple[int, int]]:
    """(start, end) spans of at most max_words words, cut at the last sentence or line end."""
    words = [m.span() for m in re.finditer(r"\S+", text)]
    spans, first, cut = [], 0, None
    for i, (s, e) in enumerate(words):
        if i - first == max_words:
            j = cut if cut is not None and cut > first else i
            spans.append((words[first][0], words[j - 1][1]))
            first, cut = j, None
        if re.search(r"[.!?:;»«\"“”]$", text[s:e]) or text[e : e + 1] == "\n":
            cut = i + 1
    if first < len(words):
        spans.append((words[first][0], words[-1][1]))
    return spans


def locate(text: str, forms: list[str]) -> list[tuple[int, int, str]]:
    """Every occurrence of the forms: longest first, never overlapping, whole words, case endings allowed."""
    found = []
    for form in {f for f in forms if f.strip()}:
        pattern = rf"(?<!\w){re.escape(form)}(?:e?s|e?n)?(?!\w)"
        found += [(m.start(), m.end(), form) for m in re.finditer(pattern, text)]
    taken, kept = [False] * len(text), []
    for s, e, form in sorted(found, key=lambda x: (x[0] - x[1], x[0])):
        if not any(taken[s:e]):
            taken[s:e] = [True] * (e - s)
            kept.append((s, e, form))
    return sorted(kept)


print(f"{len(related)} related articles, {related['text'].str.split().str.len().sum():,} words")

/home/ryounis/Documents/Zurich/ETHZ/semester-project/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


473 related articles, 349,282 words


In [18]:
SPACY_MODEL = "de_core_news_lg"
SPACY_TYPES = {"PER": "person", "ORG": "organization"}


def run_spacy(frame: pd.DataFrame) -> pd.DataFrame:
    nlp = spacy.load(SPACY_MODEL)
    rows = [
        (i, "spacy", e.start_char, e.end_char, e.text, SPACY_TYPES.get(e.label_, "other"), e.label_, np.nan)
        for i, doc in zip(frame["id"], nlp.pipe(frame["text"], batch_size=16))
        for e in doc.ents
    ]
    return pd.DataFrame(rows, columns=COLUMNS)


spacy_found = cached("spacy", [SPACY_MODEL, spacy.load(SPACY_MODEL).meta["version"]], run_spacy)
print(f"spaCy: {len(spacy_found):,} mentions")

spaCy: 25,946 mentions


In [19]:
GLINER_MODEL = "urchade/gliner_multi-v2.1"
GLINER_LABELS = ["person", "organization"]
GLINER_THRESHOLD = 0.5
GLINER_MAX_WORDS = 150  # GLiNER reads at most 384 of its tokens


def run_gliner(frame: pd.DataFrame) -> pd.DataFrame:
    model = GLiNER.from_pretrained(GLINER_MODEL)
    pieces = [(i, s, text[s:e]) for i, text in zip(frame["id"], frame["text"])
              for s, e in chunks(text, GLINER_MAX_WORDS)]
    found = model.inference([p[2] for p in pieces], GLINER_LABELS,
                            threshold=GLINER_THRESHOLD, batch_size=16)
    rows = [(i, "gliner", offset + p["start"], offset + p["end"], p["text"], p["label"], p["label"], p["score"])
            for (i, offset, _), predictions in zip(pieces, found) for p in predictions]
    return pd.DataFrame(rows, columns=COLUMNS)


gliner_found = cached("gliner", [GLINER_MODEL, GLINER_LABELS, GLINER_THRESHOLD, GLINER_MAX_WORDS], run_gliner)
print(f"GLiNER: {len(gliner_found):,} mentions")

GLiNER: 14,290 mentions


In [20]:
TAG_SYSTEM = """You mark named entities in one piece of a Swiss German-language news article.

Copy the piece exactly, character for character, and wrap every mention of a person or \
an organization in a tag: <e id="P1">...</e> for persons, <e id="O1">...</e> for \
organizations. Change nothing else: no corrections, no added or removed characters, \
spaces, line breaks or punctuation. Output only the tagged piece.

Same entity, same id. You get the table of entities already found earlier in the \
article: reuse those ids, also when the entity is written differently ("Moser" after \
"Tiana Angelina Moser", "des Gewerbeverbands" after "Gewerbeverband"). A new entity gets \
the next free number: P for persons, O for organizations.

What to tag:
- full names, surnames or first names alone, abbreviations, and names in any \
grammatical form;
- a party or organization abbreviation inside a compound word, tagged on its own: \
<e id="O1">FDP</e>-Chefin, <e id="O2">SVP</e>-Initiative;
- organizations include political parties, government bodies (Bundesrat, Parlament, \
Nationalrat), companies, associations, unions, media outlets and international bodies \
such as the EU.

What not to tag: pronouns; role words without a name ("die Bundesrätin"); a title before \
a name, which stays outside the tag (Bundesrätin <e id="P2">Karin Keller-Sutter</e>); \
places; events; laws and ballot proposals; the journalist who wrote the article.

Tags are never nested."""
TAG = re.compile(r'<e id="([PO]\d+)">(.*?)</e>', re.DOTALL)
TAG_PIECE_WORDS = 400
TAG_RETRIES = 2
TAG_MAX_LOST = 0.5  # a copy that skips more than this is retried; the best attempt is then patched
TAG_PATCH_DEPTH = 2  # skipped passages are re-tagged, and their skips once more
TAG_SHA = config_sha([MODEL, TAG_SYSTEM, TAG_PIECE_WORDS, TAG_MAX_LOST, TAG_PATCH_DEPTH, "adaptive", "medium"])
TAG_CACHE = INTERIM / "a631_ner_haiku_tags.jsonl"


def untag(tagged: str) -> tuple[str, list[tuple[int, int, str]]]:
    """Plain text and (start, end, entity id) of every tag, positions in the plain text."""
    plain, spans, last = [], [], 0
    for m in TAG.finditer(tagged):
        plain.append(tagged[last : m.start()])
        start = sum(map(len, plain))
        plain.append(m.group(2))
        spans.append((start, start + len(m.group(2)), m.group(1)))
        last = m.end()
    plain.append(tagged[last:])
    return "".join(plain), spans


def align(piece: str, plain: str, spans) -> tuple[list, int, list[tuple[int, int]]]:
    """Carry tags from the model's copy back onto the original piece.

    The model sometimes edits while copying (a corrected case ending, an added
    word, a skipped headline). Aligning copy and original maps each tag to its
    original position; a tag is kept only if its text sits unchanged inside one
    aligned block, so every kept mention is verified against the original. Text
    the model adds matches nothing in the original, so no tag can come from it.
    Text it skips is what costs mentions: those stretches come back as gaps,
    widened to whole words, so they can be tagged in a second pass.
    Returns the kept spans, the number of dropped tags and the gaps.
    """
    blocks = difflib.SequenceMatcher(None, plain, piece, autojunk=False).get_matching_blocks()
    kept = []
    for s, e, eid in spans:
        block = next((b for b in blocks if b.a <= s and e <= b.a + b.size), None)
        if block is not None:
            kept.append((block.b + s - block.a, block.b + e - block.a, eid))
    gaps, covered = [], 0
    for b in sorted(blocks, key=lambda b: b.b):
        if b.b > covered:
            gaps.append((covered, b.b))
        covered = max(covered, b.b + b.size)
    widened = []
    for a, z in gaps:
        gap = piece[a:z]
        a, z = a + len(gap) - len(gap.lstrip()), z - len(gap) + len(gap.rstrip())
        if a >= z:
            continue                    # only whitespace was lost
        a = max(piece.rfind(" ", 0, a), piece.rfind("\n", 0, a)) + 1
        z = min([i for i in (piece.find(" ", z), piece.find("\n", z)) if i != -1] or [len(piece)])
        if not re.search(r"\w", piece[a:z]):
            continue
        if widened and a <= widened[-1][1]:
            widened[-1] = (widened[-1][0], max(z, widened[-1][1]))
        else:
            widened.append((a, z))
    return kept, len(spans) - len(kept), widened


def table_lines(names: dict[str, str]) -> str:
    if not names:
        return "(none yet)"
    order = sorted(names, key=lambda i: (i[0], int(i[1:])))
    return "\n".join(f"{i} = {names[i]}" for i in order)


def update_names(names: dict[str, str], text: str, spans) -> None:
    """An entity's name in the table is the longest form it has been tagged with."""
    for s, e, eid in spans:
        if len(text[s:e]) > len(names.get(eid, "")):
            names[eid] = text[s:e]


async def tag_piece(client, piece: str, names: dict[str, str], depth: int = 0):
    """Tag one piece. Passages the copy skipped are tagged again on their own.

    Returns the spans, the token and repair counts, and an error or None.
    """
    used = dict.fromkeys(["input_tokens", "output_tokens", "attempts", "dropped_tags",
                          "patched_chars", "missing_chars"], 0)
    best, best_lost = None, 0
    for _ in range(1 + TAG_RETRIES):
        used["attempts"] += 1
        response = await client.messages.create(
            model=MODEL, max_tokens=16_000, thinking={"type": "adaptive"},
            output_config={"effort": "medium"}, system=TAG_SYSTEM,
            messages=[{"role": "user", "content":
                       f"Entities already found in this article:\n{table_lines(names)}\n\n"
                       f"Piece:\n{piece}"}],
        )
        used["input_tokens"] += response.usage.input_tokens
        used["output_tokens"] += response.usage.output_tokens
        if response.stop_reason != "end_turn":
            continue
        tagged = next((b.text for b in response.content if b.type == "text"), "")
        attempt = align(piece, *untag(tagged))
        lost = sum(z - a for a, z in attempt[2])
        if best is None or lost < best_lost:
            best, best_lost = attempt, lost
        if lost <= TAG_MAX_LOST * len(piece):
            break                       # good enough: whatever it skipped is patched below
    if best is None:
        return [], used, "no finished answer for the piece"
    spans, dropped, gaps = best         # the best copy; its skipped passages are patched below
    used["dropped_tags"] += dropped
    if depth == TAG_PATCH_DEPTH:
        used["missing_chars"] += sum(z - a for a, z in gaps)
        return spans, used, None
    local = dict(names)
    update_names(local, piece, spans)
    for a, z in gaps:
        sub, sub_used, error = await tag_piece(client, piece[a:z], local, depth + 1)
        for k in sub_used:
            used[k] += sub_used[k]
        if error:
            return [], used, error
        spans = [x for x in spans if x[1] <= a or x[0] >= z] + [(a + s, a + e, i) for s, e, i in sub]
        used["patched_chars"] += z - a
    return sorted(spans), used, None


async def tag_article(client, article_id: str, text: str) -> dict:
    """Tag an article piece by piece, carrying the entity table forward."""
    names, mentions = {}, []
    used = dict.fromkeys(["input_tokens", "output_tokens", "attempts", "dropped_tags", "patched_chars", "missing_chars"], 0)
    for start, end in chunks(text, TAG_PIECE_WORDS):
        spans, piece_used, error = await tag_piece(client, text[start:end], names)
        for k in piece_used:
            used[k] += piece_used[k]
        if error:
            return {"id": article_id, "sha": TAG_SHA, "error": error, **used}
        spans = [(start + s, start + e, eid) for s, e, eid in spans]
        update_names(names, text, spans)
        mentions += spans
    return {"id": article_id, "sha": TAG_SHA, "entities": names, "mentions": mentions, **used}


def tag_results() -> dict[str, dict]:
    if not TAG_CACHE.exists():
        return {}
    rows = (json.loads(line) for line in TAG_CACHE.read_text().splitlines())
    return {r["id"]: r for r in rows if r["sha"] == TAG_SHA}  # a later line for the same id wins


async def tag_missing() -> list[str]:
    done = tag_results()
    todo = related[[i not in done or "error" in done[i] for i in related["id"]]]
    if todo.empty:
        return []
    gate = asyncio.Semaphore(CONCURRENCY)

    async def one(client, article_id, text, sink):
        async with gate:
            try:
                row = await tag_article(client, article_id, text)
            except (anthropic.APIStatusError, anthropic.APIConnectionError) as e:
                return f"{article_id}: {type(e).__name__}"
        sink.write(json.dumps(row) + "\n")
        sink.flush()
        return f"{article_id}: {row['error']}" if "error" in row else None

    async with anthropic.AsyncAnthropic() as client:
        with TAG_CACHE.open("a") as sink:
            async with asyncio.TaskGroup() as tg:
                tasks = [tg.create_task(one(client, i, t, sink)) for i, t in zip(todo["id"], todo["text"])]
    return [t.result() for t in tasks if t.result()]


failures = await tag_missing()
tagged = tag_results()
for failure in failures:
    print("failed:", failure)
assert all("error" not in tagged[i] for i in related["id"]), "some articles could not be tagged"

def fill_repeats(text: str, spans) -> list[tuple[int, int, str]]:
    """Untagged occurrences of a form tagged elsewhere in the article, with that form's id.

    Longest form first and never overlapping a tag or another fill, so an untagged
    "Tiana Angelina Moser" is filled once, not also as "Moser". A form tagged with
    several ids takes the one it carries most often.
    """
    taken = [False] * len(text)
    for s, e, _ in spans:
        taken[s:e] = [True] * (e - s)
    ids: dict[str, collections.Counter] = {}
    for s, e, eid in spans:
        ids.setdefault(text[s:e], collections.Counter())[eid] += 1
    filled = []
    for form in sorted(ids, key=len, reverse=True):
        eid = ids[form].most_common(1)[0][0]
        for m in re.finditer(rf"(?<!\w){re.escape(form)}(?!\w)", text):
            if not any(taken[m.start() : m.end()]):
                taken[m.start() : m.end()] = [True] * (m.end() - m.start())
                filled.append((m.start(), m.end(), eid))
    return sorted(filled)


rows = []
for i, text in zip(related["id"], related["text"]):
    spans = [tuple(x) for x in tagged[i]["mentions"]]
    rows += [(i, s, e, eid, False) for s, e, eid in spans]
    rows += [(i, s, e, eid, True) for s, e, eid in fill_repeats(text, spans)]
haiku_found = pd.DataFrame(
    [(i, "haiku", s, e, texts_by_id[i][s:e], "person" if eid[0] == "P" else "organization", eid, np.nan, eid, filled)
     for i, s, e, eid, filled in rows],
    columns=[*COLUMNS, "entity", "filled"])

results = [tagged[i] for i in related["id"]]
used_in, used_out = sum(r["input_tokens"] for r in results), sum(r["output_tokens"] for r in results)
print(f"Haiku: {len(haiku_found):,} mentions of {sum(len(r['entities']) for r in results):,} entities "
      f"({sum(r['attempts'] for r in results):,} calls)")
print(f"copy repairs: {sum(r['patched_chars'] for r in results):,} skipped characters tagged in a second pass, "
      f"{sum(r['missing_chars'] for r in results):,} left untagged, {sum(r['dropped_tags'] for r in results)} tags dropped")
print(f"filled in: {haiku_found['filled'].sum():,} untagged repeats of a tagged form "
      f"({100 * haiku_found['filled'].mean():.1f}% of Haiku's mentions, column `filled`)")
print(f"tokens {used_in:,} in, {used_out:,} out ≈ ${used_in / 1e6 * 0.10 + used_out / 1e6 * 0.50:.2f}")

Haiku: 14,255 mentions of 5,010 entities (1,427 calls)
copy repairs: 21,080 skipped characters tagged in a second pass, 36 left untagged, 400 tags dropped
filled in: 984 untagged repeats of a tagged form (6.9% of Haiku's mentions, column `filled`)
tokens 2,203,195 in, 1,944,450 out ≈ $1.19


### Checks

Every stored position must point at the stored mention text, and the per-article
dictionaries must be an exact tally of the mention rows.

In [21]:
mentions = pd.concat([spacy_found, gliner_found, haiku_found], ignore_index=True)
texts = dict(zip(related["id"], related["text"]))
assert all(texts[r.id][r.start:r.end] == r.mention for r in mentions.itertuples()), "a position is off"
assert mentions.groupby("model")["id"].nunique().max() <= len(related)

actors = mentions[mentions["type"] != "other"]


def entity_counts(model: str) -> dict[str, dict[str, int]]:
    """{article id: {mention: count}} for one model, persons and organizations only."""
    found = actors[actors["model"] == model]
    return {i: g["mention"].value_counts().to_dict() for i, g in found.groupby("id")}


counts = {m: entity_counts(m) for m in ("spacy", "gliner", "haiku")}
for model, per_article in counts.items():
    assert sum(sum(d.values()) for d in per_article.values()) == (actors["model"] == model).sum()
print("positions and tallies check out")

positions and tallies check out


### First look

In [22]:
(actors.groupby("model")
       .agg(mentions=("id", "size"),
            persons=("type", lambda t: (t == "person").sum()),
            organizations=("type", lambda t: (t == "organization").sum()),
            distinct_names=("mention", "nunique"),
            articles_with_none=("id", lambda i: len(related) - i.nunique()))
       .assign(per_article=lambda t: (t.mentions / len(related)).round(1)))

,mentions,persons,organizations,distinct_names,articles_with_none,per_article
model,,,,,,
gliner,14290,4675,9615,2591,1,30.2
haiku,14255,4858,9397,1702,2,30.1
spacy,13560,5122,8438,2436,0,28.7


In [23]:
def side_by_side(article_id: str) -> pd.DataFrame:
    """Mention counts in one article, one column per model."""
    one = actors[actors["id"] == article_id]
    return (one.pivot_table(index="mention", columns="model", values="start", aggfunc="count", fill_value=0)
               .reindex(columns=["spacy", "gliner", "haiku"], fill_value=0)
               .sort_values(["haiku", "spacy", "gliner"], ascending=False))


example = related.sample(1, random_state=7).iloc[0]
print(f"{example['id']}  {example['medium_name']}")
side_by_side(example["id"])

40741618  Tages-Anzeiger


model,spacy,gliner,haiku
mention,,,
Bundesrat,0,10,10
EU,7,8,8
Moser,6,4,6
FDP,1,1,3
Pfister,1,2,2
CVP,1,1,1
Christian Rathgeb,1,1,1
Gerhard Pfister,1,1,1
Gewerkschaften,1,1,1


## Read a random article

Draws one article from the vote slice and prints it in full. Narrow the draw
with `relevance="related"` / `"not_related"` and/or an outlet code, e.g.
`read_random("not_related", outlet="SGTO")`.

**Clear this cell's output before committing.** Article text is licensed corpus
content and must not enter git.

In [ ]:
def read_random(relevance: str | None = None, outlet: str | None = None) -> None:
    pool = vote
    if relevance is not None and "relevance" not in vote:
        raise RuntimeError("vote has no 'relevance' column: run the section 4 cells first "
                           "(labels come from the cache, no API call)")
    if relevance is not None:
        pool = pool[pool["relevance"] == relevance]
    if outlet is not None:
        pool = pool[pool["medium_code"] == outlet]
    if pool.empty:
        print("no article matches")
        return
    show(int(pool.sample(1).index[0]))  # vote has a fresh RangeIndex, so label == position


read_random("related")